# VDM for generative modeling

This notebook implements the Variational Diffusion Model (VDM), a special case of the SDE framework from notebook 02 with a fixed (non-neural) variational distribution.

The variational marginal is restricted to
$$
q_t(x|y) = \mathcal{N}\!\left(x \mid \alpha(t)\,y,\, \beta^2(t)\right),
$$
where $\alpha(t)$ and $\beta(t)$ are **fixed** scalar functions of $t$ satisfying variance preservation $\alpha^2(t)+\beta^2(t)=1$. Here **$t=0$ is the prior** ($\alpha(0)=0$, $\beta(0)=1$, so $q_0(x|y)=\mathcal{N}(0,1)$) and **$t=1$ is the data** ($\alpha(1)\approx 1$, $\beta(1)\approx 0$), consistent with notebooks 01–03.

The SNR schedule is linear in log space with learnable endpoints:
$$
\log \mathrm{SNR}(t) = (1-t)\,\log \mathrm{SNR}(0) + t\,\log \mathrm{SNR}(1),
\qquad
\alpha(t) = \sqrt{\frac{\mathrm{SNR}(t)}{1+\mathrm{SNR}(t)}},
\quad
\beta(t) = \frac{1}{\sqrt{1+\mathrm{SNR}(t)}}.
$$
with $\log\mathrm{SNR}(0)\ll 0$ (noise) and $\log\mathrm{SNR}(1)\gg 0$ (data).

The generative drift is parameterised via a **signal prediction network** $\widehat{y}(x_t, t)$:
$$
\overrightarrow{f}(x,t) = \overrightarrow{g}(x,\,\widehat{y}(x,t),\,t),
$$
and the ELBO drift term simplifies to the SNR-weighted objective
$$
\mathcal{D} = \frac{1}{2}\int_0^1 \left|\frac{\partial\,\mathrm{SNR}(t)}{\partial t}\right|
\mathbb{E}_{q_t(x|y)}\!\left[\|y - \widehat{y}(x_t,t)\|^2\right]\mathrm{d}t.
$$

The full ELBO is
$$
\log p(y) \ge
\underbrace{\mathbb{E}_{q_1}[\log p(y|x_1)]}_{\text{likelihood at }t=1}
- \underbrace{\mathrm{KL}(q_0(x|y)\|\,p_0(x))}_{=\,0\text{ exactly (both }=\mathcal{N}(0,1))}
- \mathcal{D}.
$$

Generation runs the **forward generative SDE** from $x_0 \sim \mathcal{N}(0,1)$ at $t=0$ to $x_1 \approx y$ at $t=1$.

## Setup

In [ ]:
import os, sys

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    os.system("wget -q https://raw.githubusercontent.com/olewinther/generative-ode-sde/main/utils.py")
else:
    for path in ['..', '.']:
        if os.path.exists(os.path.join(path, 'utils.py')):
            sys.path.insert(0, os.path.abspath(path))
            break

from utils import *

## GPU

In [ ]:
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print('GPU State:', device)

## SNR schedule

The schedule holds $\log\mathrm{SNR}(0)$ and $\log\mathrm{SNR}(1)$ (learnable or fixed) and derives $\alpha(t)$, $\beta(t)$, and the diffusion coefficient
$$
\sigma^2(t) = \alpha^2(t)\,\frac{\partial}{\partial t}\!\left[\frac{\beta^2(t)}{\alpha^2(t)}\right]
= \bigl(\log\mathrm{SNR}(1)-\log\mathrm{SNR}(0)\bigr)\,\beta^2(t) > 0.
$$
Since $\log\mathrm{SNR}(1)>\log\mathrm{SNR}(0)$ in our convention, $\sigma^2(t)$ is always positive.

In [ ]:
import torch.nn as nn

class SNRSchedule(nn.Module):
    """
    Linear log-SNR schedule with variance preservation.
    log_snr_0: log SNR at t=0 (noise end, low SNR).
    log_snr_1: log SNR at t=1 (data end, high SNR).
    Set trainable=True to learn the endpoints jointly with the model.
    """
    def __init__(self, log_snr_0=-10.0, log_snr_1=10.0, trainable=False):
        super().__init__()
        for name, val in [('log_snr_0', log_snr_0), ('log_snr_1', log_snr_1)]:
            buf = torch.tensor(float(val))
            if trainable:
                setattr(self, name, nn.Parameter(buf))
            else:
                self.register_buffer(name, buf)

    def log_snr(self, t):
        return (1 - t) * self.log_snr_0 + t * self.log_snr_1

    def snr(self, t):
        return torch.exp(self.log_snr(t))

    def alpha(self, t):
        return torch.sqrt(self.snr(t) / (1 + self.snr(t)))

    def beta(self, t):
        return torch.sqrt(1 / (1 + self.snr(t)))

    def sigma_sq(self, t):
        # σ²(t) = (log_snr_1 - log_snr_0) · β²(t)  [always > 0 since log_snr_1 > log_snr_0]
        return (self.log_snr_1 - self.log_snr_0) * self.beta(t) ** 2

    def sigma(self, t):
        return torch.sqrt(self.sigma_sq(t))

    def d_log_alpha_dt(self, t):
        # d/dt log α(t) = (log_snr_1 - log_snr_0) / 2 · β²(t)  [always > 0]
        return 0.5 * (self.log_snr_1 - self.log_snr_0) * self.beta(t) ** 2

    def d_snr_dt(self, t):
        # d/dt SNR(t) = (log_snr_1 - log_snr_0) · SNR(t)  [always > 0]
        return (self.log_snr_1 - self.log_snr_0) * self.snr(t)


class VDMAlpha(nn.Module):
    """Variational mean: alpha(y, t) = schedule.alpha(t) * y"""
    def __init__(self, schedule):
        super().__init__()
        self.schedule = schedule

    def forward(self, y, t):
        return self.schedule.alpha(t) * y


class VDMBeta(nn.Module):
    """Variational std: beta(y, t) = schedule.beta(t)"""
    def __init__(self, schedule):
        super().__init__()
        self.schedule = schedule

    def forward(self, y, t):
        return self.schedule.beta(t).expand_as(y)


class VDMSigma(nn.Module):
    """Diffusion coefficient: sigma(x, t) = schedule.sigma(t)"""
    def __init__(self, schedule):
        super().__init__()
        self.schedule = schedule

    def forward(self, x, t):
        return self.schedule.sigma(t).expand_as(x)

## Signal prediction network

In [ ]:
class SignalPredictionNetwork(nn.Module):
    """Predicts y from noisy observation x_t and time t."""
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 64), nn.Tanh(),
            nn.Linear(64, 64), nn.Tanh(),
            nn.Linear(64, 1),
        )

    def forward(self, x, t):
        return self.net(torch.cat([x, t], dim=-1))

## Visualize schedule and learned signal prediction

In [ ]:
def plot_schedule(schedule):
    t_grid = torch.linspace(0, 1, 200).unsqueeze(1)
    with torch.no_grad():
        alpha_vals = [schedule.alpha(t).item() for t in t_grid]
        beta_vals  = [schedule.beta(t).item()  for t in t_grid]
        snr_vals   = [schedule.snr(t).item()   for t in t_grid]

    fig, axes = plt.subplots(3, 1, figsize=(10, 9))
    axes[0].plot(t_grid.numpy(), alpha_vals)
    axes[0].set_title(r"$\alpha(t)$"); axes[0].set_xlabel("t"); axes[0].set_ylabel(r"$\alpha(t)$")
    axes[1].plot(t_grid.numpy(), beta_vals)
    axes[1].set_title(r"$\beta(t)$");  axes[1].set_xlabel("t"); axes[1].set_ylabel(r"$\beta(t)$")
    axes[2].semilogy(t_grid.numpy(), snr_vals)
    axes[2].set_title("SNR(t)");        axes[2].set_xlabel("t"); axes[2].set_ylabel("SNR(t)")
    plt.tight_layout(); plt.show()


def plot_y_hat(y_hat_net, schedule, num_samples=3):
    t_grid = torch.linspace(0, 1, 100).unsqueeze(1)
    x_samples = torch.randn(num_samples, 1)

    fig, ax = plt.subplots(figsize=(10, 3))
    for s in x_samples:
        with torch.no_grad():
            vals = [y_hat_net(s, t.expand_as(s)).item() for t in t_grid]
        ax.plot(t_grid.numpy(), vals, label=f"x={s.item():.2f}")
    ax.set_title(r"$\widehat{y}(x,t)$"); ax.set_xlabel("t"); ax.set_ylabel(r"$\widehat{y}$"); ax.legend()
    plt.tight_layout(); plt.show()

## VDM ELBO and training loop

The ELBO boundary terms match the convention in notebook 02: the **likelihood is evaluated at $t=1$** (data end) and the **KL is at $t=0$** (noise end). Since $q_0(x|y)=\mathcal{N}(0,1)=p_0(x)$ exactly, the KL vanishes identically — it is retained here for reporting purposes only.

In [ ]:
def compute_elbo_vdm(y_hat_net, vdm_alpha, vdm_beta, schedule, prior, likelihood, y, t_sample):
    # 1. Likelihood term at t=1: E_{q_1(x|y)}[log p(y|x_1)]
    q_x1 = Variational(vdm_alpha, vdm_beta, y, torch.ones_like(t_sample))
    x_1 = q_x1.sample()
    likelihood_term = likelihood.log_prob(y, x_1)

    # 2. KL at t=0: KL(q_0(x|y) || p_0(x))  [= 0 exactly since both are N(0,1)]
    q_x0 = Variational(vdm_alpha, vdm_beta, y, torch.zeros_like(t_sample))
    x_0 = q_x0.sample()
    kl_divergence = q_x0.log_prob(x_0) - prior.log_prob(x_0)

    # 3. SNR-weighted signal prediction loss
    eps = torch.randn_like(y)
    x_t = schedule.alpha(t_sample) * y + schedule.beta(t_sample) * eps
    y_hat = y_hat_net(x_t, t_sample)
    drift_term = 0.5 * torch.abs(schedule.d_snr_dt(t_sample)) * (y - y_hat) ** 2

    elbo = likelihood_term - kl_divergence - drift_term
    return elbo.mean(), likelihood_term.mean(), kl_divergence.mean(), drift_term.mean()


def training_loop_vdm(y_hat_net, vdm_alpha, vdm_beta, schedule, prior, likelihood,
                      data_loader, validation_data, n_epochs=1000, lr=1e-3):
    from collections import deque
    trainable = (list(y_hat_net.parameters()) + list(schedule.parameters()) +
                 list(likelihood.beta.parameters()))
    optimizer = torch.optim.Adam(trainable, lr=lr)
    train_history, val_history = deque(maxlen=5), deque(maxlen=5)

    for epoch in range(n_epochs):
        total_loss = 0.0
        for y_batch in data_loader:
            optimizer.zero_grad()
            t_sample = torch.rand(y_batch.size())
            eps = torch.randn_like(y_batch)
            x_t = schedule.alpha(t_sample) * y_batch + schedule.beta(t_sample) * eps
            y_hat = y_hat_net(x_t, t_sample)
            loss = (0.5 * torch.abs(schedule.d_snr_dt(t_sample)) * (y_batch - y_hat) ** 2).mean()
            total_loss += loss.item()
            loss.backward()
            optimizer.step()

        if epoch % 50 == 0 or epoch == n_epochs - 1:
            t_val = torch.rand(validation_data.size())
            elbo_val, ll, kl, dm = compute_elbo_vdm(
                y_hat_net, vdm_alpha, vdm_beta, schedule, prior, likelihood,
                validation_data, t_val)
            train_cur = total_loss / len(data_loader)
            val_cur = elbo_val.item()
            train_history.append(train_cur)
            val_history.append(val_cur)
            avg_train = sum(train_history) / len(train_history)
            avg_val   = sum(val_history)   / len(val_history)
            print(f"Epoch {epoch:4d} | train {train_cur:.4f} (avg {avg_train:.4f}) | "
                  f"val {val_cur:.4f} (avg {avg_val:.4f}) | "
                  f"ll={ll:.4f}, kl={kl:.4f}, dm={dm:.4f} | "
                  f"log_snr_0={schedule.log_snr_0.item():.2f} "
                  f"log_snr_1={schedule.log_snr_1.item():.2f}")

    return y_hat_net, schedule

## Generation (forward generative SDE) and visualization

Generation runs the **forward generative SDE** from $x_0\sim\mathcal{N}(0,1)$ at $t=0$ to $x_1\approx y$ at $t=1$:
$$
dx = \left[\frac{\partial\log\alpha}{\partial t}\cdot x + \sigma^2(t)\,\nabla_x\log p_t(x)\right]dt + \sigma(t)\,d\widetilde{W},
$$
with score approximation $\nabla_x\log p_t(x) \approx (\alpha(t)\widehat{y}(x,t)-x)/\beta^2(t)$.

Combined into a single drift:
$$
f_{\text{gen}}(x,t) = \frac{\partial\log\alpha}{\partial t}\cdot x + \sigma^2(t)\cdot\frac{\alpha(t)\widehat{y}(x,t)-x}{\beta^2(t)}.
$$

In [ ]:
def generate_vdm(y_hat_net, schedule, n_samples, n_steps=100):
    """Forward generative SDE from t=0 (noise) to t=1 (data) via Euler-Maruyama."""
    x = torch.randn(n_samples, 1)
    t_grid = torch.linspace(0.0, 1.0, n_steps + 1)

    for i in range(n_steps):
        t_i = t_grid[i].expand(n_samples, 1)
        dt  = (t_grid[i + 1] - t_grid[i]).item()  # positive

        with torch.no_grad():
            y_hat = y_hat_net(x, t_i)

        alpha_t       = schedule.alpha(t_i)
        beta_t        = schedule.beta(t_i)
        sigma_sq_t    = schedule.sigma_sq(t_i)
        d_log_alpha_t = schedule.d_log_alpha_dt(t_i)

        score     = (alpha_t * y_hat - x) / beta_t ** 2
        gen_drift = d_log_alpha_t * x + sigma_sq_t * score

        noise = torch.sqrt(torch.tensor(dt) * sigma_sq_t) * torch.randn_like(x)
        x = x + gen_drift * dt + noise

    return x


def visualize_vdm_paths_and_marginals(validation_data, y_hat_net, schedule, prior, n_steps=100):
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    t_plot = torch.linspace(0, 1, 51)

    # --- Forward paths: q(x_t|y) = alpha(t)*y + beta(t)*eps, noise → data ---
    ax = axes[0]
    for y_i in validation_data[:6]:
        path = []
        for t in t_plot:
            t_i = t.view(1, 1)
            x_t = schedule.alpha(t_i) * y_i + schedule.beta(t_i) * torch.randn(1, 1)
            path.append(x_t.item())
        ax.plot(t_plot.numpy(), path, alpha=0.6)
    ax.set_xlabel('t'); ax.set_ylabel('x'); ax.set_title('Forward paths (noise → data)')

    # --- Generative paths: forward generative SDE from t=0 to t=1 ---
    ax = axes[1]
    n_paths = 6
    x_curr = prior.sample(n_paths)
    t_fwd = torch.linspace(0.0, 1.0, 51)
    trajs = [[(t_fwd[0].item(), x_curr[j].item())] for j in range(n_paths)]

    for i in range(50):
        t_i = t_fwd[i].expand(n_paths, 1)
        dt  = (t_fwd[i + 1] - t_fwd[i]).item()
        with torch.no_grad():
            y_hat = y_hat_net(x_curr, t_i)
        alpha_t       = schedule.alpha(t_i)
        beta_t        = schedule.beta(t_i)
        sigma_sq_t    = schedule.sigma_sq(t_i)
        d_log_alpha_t = schedule.d_log_alpha_dt(t_i)
        score     = (alpha_t * y_hat - x_curr) / beta_t ** 2
        gen_drift = d_log_alpha_t * x_curr + sigma_sq_t * score
        noise = torch.sqrt(torch.tensor(dt) * sigma_sq_t) * torch.randn_like(x_curr)
        x_curr = x_curr + gen_drift * dt + noise
        for j in range(n_paths):
            trajs[j].append((t_fwd[i + 1].item(), x_curr[j].item()))

    for j in range(n_paths):
        ts, xs = zip(*trajs[j])
        ax.plot(ts, xs, alpha=0.6)
    ax.set_xlabel('t'); ax.set_ylabel('x'); ax.set_title('Generative paths (noise → data)')

    # --- Marginals: data vs generated ---
    ax = axes[2]
    with torch.no_grad():
        generated = generate_vdm(y_hat_net, schedule, n_samples=len(validation_data), n_steps=n_steps)
    ax.hist(validation_data.numpy().flatten(), bins=60, density=True, alpha=0.5, label='data')
    ax.hist(generated.numpy().flatten(),       bins=60, density=True, alpha=0.5, label='VDM')
    ax.set_title('Data vs generated'); ax.legend()

    plt.tight_layout(); plt.show()

## Create training and validation data

In [ ]:
torch.manual_seed(42)
np.random.seed(42)

n_samples, n_val = 1000, 8000

# Choose distribution: 'gaussian', 'laplace', 'laplace_mixture'
training_set_dist = 'laplace_mixture'

if training_set_dist == 'gaussian':
    params = {'mean': torch.tensor(-1.0), 'std': torch.tensor(2.0)}
elif training_set_dist == 'laplace':
    params = {'loc': torch.tensor(0.0), 'scale': torch.tensor(1.0)}
elif training_set_dist == 'laplace_mixture':
    params = {'k': 5, 'spacing': 4.0, 'scale': torch.tensor(1.0)}

training_set = TrainingSetWithLogLikelihood(training_set_dist, params)
training_data, ell_train = training_set.generate_training_data(n_samples)
validation_data, ell_val = training_set.generate_training_data(n_val)
print(f"{training_set_dist} | true log-likelihood: train={ell_train:.4f}, val={ell_val:.4f}")

## Run training

In [ ]:
data_loader = torch.utils.data.DataLoader(training_data, batch_size=125, shuffle=True)

# --- SNR schedule ---
# log_snr_0 << 0: low SNR at t=0 (noise); log_snr_1 >> 0: high SNR at t=1 (data)
# trainable=False: fixed schedule (standard VDM)
# trainable=True:  learn the SNR endpoints jointly with the model
schedule = SNRSchedule(log_snr_0=-10.0, log_snr_1=10.0, trainable=False)

# --- Networks ---
y_hat_net  = SignalPredictionNetwork()
vdm_alpha  = VDMAlpha(schedule)
vdm_beta   = VDMBeta(schedule)
vdm_sigma  = VDMSigma(schedule)

# --- Prior: N(0,1) at t=0 ---
prior = Prior(gaussian_sample, gaussian_log_pdf)

# --- Likelihood: p(y|x_1) = N(y|x_1, delta^2) ---
delta = Beta(initial_beta=1e-1, trainable=True)
likelihood = Likelihood(beta=delta)

print("Before training:")
plot_schedule(schedule)
plot_y_hat(y_hat_net, schedule)
visualize_vdm_paths_and_marginals(validation_data, y_hat_net, schedule, prior)

y_hat_net, schedule = training_loop_vdm(
    y_hat_net, vdm_alpha, vdm_beta, schedule, prior, likelihood,
    data_loader, validation_data, n_epochs=1000, lr=1e-3,
)

print("After training:")
plot_schedule(schedule)
plot_y_hat(y_hat_net, schedule)
visualize_vdm_paths_and_marginals(validation_data, y_hat_net, schedule, prior)